##  Gridded Model Verification

This script verifies output from a ML-based foundation model versus a
traditional NWP system for the atmospheric system. The defaults set at the top of
this script are tailored to the Alps-Clariden HPC system at CSCS.
- The NWP-model is called COSMO-E and is initialised with the ensemble mean of the analysis. Only surface level data is available in the archive at MeteoSwiss.
- The ML-model is called Neural-LAM and is initialised with the deterministic analysis.
- The Ground Truth is the same deterministic analysis as was used to train the ML-model.
- The boundary data for both models is IFS HRES from ECMWF, where the NWP-model got 6 hourly boundary updates and the ML model 12 hourly.

For more info about the COSMO model see:
- https://www.cosmo-model.org/content/model/cosmo/coreDocumentation/cosmo_io_guide_6.00.pdf
- https://www.research-collection.ethz.ch/handle/20.500.11850/720460

In [1]:
import random
import os
from pathlib import Path

import PIL
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import dask
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


/users/sadamov/miniforge3/envs/neural-lam/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


**--------> Enter all your user settings in the cell below. <--------**

In [ ]:
### DEFAULTS ###
# This config will be applied to the data before any plotting. The data will be
# sliced and indexed according to the values in this config.The whole analysis and
# plotting will be done on the reduced data.

# IF YOUR DATA HAS DIFFERENT DIMENSIONS OR NAMES, PLEASE ADJUST THE CELLS BELOW
# MAKE SURE THE XARRAY DATASETS LOOK OKAY BEFORE RUNNING CHAPTER 1-4

# This path should point to the data that was used to train the model (default is mdp-datastore)
PATH_GROUND_TRUTH = "cosmo.datastore.zarr"
# This path should point to the NWP forecast data in zarr format
PATH_NWP = "cosmo_e_forecast.zarr"
# This path should point to the ML forecast data in zarr format (e.g. produced by neural-lam in `eval` mode)
PATH_ML = "lam_model_forecasts/preds_7_19_margin_interior_lr_0001_ar_12.zarr"
# This path should point to the boundary data in zarr format (default is MDP-datastore)
PATH_BOUNDARY = "ifs_7_19_margin_interior.datastore.zarr"

# elapsed forecast duration in steps for the forecast - [0] refers to the first forecast step at t+1
# this should be a list of integers
ELAPSED_FORECAST_DURATION = list(range(0, 119))

# Select specific start_times for the forecast. This is the start and end of
# a slice in xarray. The start_time is included, the end_time is excluded.
# This should be a list of two strings in the format "YYYY-MM-DDTHH:MM:SS"
# Should be handy to evaluate certain dates, e.g. for a case study of a storm
# START_TIMES = ["2019-10-31T00:00:00", "2020-10-23T13:00:00"]  # Full year
START_TIMES = ["2020-02-08T00:00:00", "2020-02-10T00:00:00"]  # Ciara/Sabine

# Select specific plot times for the forecast (will be used to create maps for all variables)
# This only affect chapter one with the plotting of the maps
# Map creation takes a lot of time so this is limited to a single time step
# Simply rerun these cells and chapter one for more time steps
PLOT_TIME = "2020-02-08T00:00:00"

# Animation details
ANIMATION_DIR = "gifs"
ANIMATION_DURATION = 200  # ms

# Selection spatial grid in projection
# This is used to slice the data to a specific region
# This is in projection of the ground truth data
# The default is the whole domain [None, None]
X = [None, None]
Y = [None, None]

# Map projection settings for plotting
# This is the projection of the ground truth data
PROJECTION = ccrs.RotatedPole(
    pole_longitude=190,
    pole_latitude=43,
    central_rotated_longitude=10,
)

# Define how variables map between different data sources

# Define here which of the variables are available in the ground truth data
# The keys are the names of the variables in the ground truth data
# The values are the conventional names, used in this notebook
VARIABLES_GROUND_TRUTH = {
    # Surface and near-surface variables
    # "T_2M": "temperature_2m",
    "U_10M": "wind_u_10m",
    # "V_10M": "wind_v_10m",
    # "PMSL": "pressure_sea_level",
    # "PS": "surface_pressure",
    # "TOT_PREC": "precipitation",
    # "ASHFL_S": "surface_sensible_heat_flux",
    # "ASOB_S": "surface_net_shortwave_radiation",
    "ATHB_S": "surface_net_longwave_radiation",
    # # Upper air variables - U component
    # "U_lev_6": "wind_u_level_6",
    # "U_lev_12": "wind_u_level_12",
    # "U_lev_20": "wind_u_level_20",
    # "U_lev_27": "wind_u_level_27",
    # "U_lev_31": "wind_u_level_31",
    # "U_lev_39": "wind_u_level_39",
    # "U_lev_45": "wind_u_level_45",
    # "U_lev_60": "wind_u_level_60",
    # # Upper air variables - V component
    # "V_lev_6": "wind_v_level_6",
    # "V_lev_12": "wind_v_level_12",
    # "V_lev_20": "wind_v_level_20",
    # "V_lev_27": "wind_v_level_27",
    # "V_lev_31": "wind_v_level_31",
    # "V_lev_39": "wind_v_level_39",
    # "V_lev_45": "wind_v_level_45",
    # "V_lev_60": "wind_v_level_60",
    # # Upper air variables - Pressure
    # "PP_lev_6": "pressure_level_6",
    # "PP_lev_12": "pressure_level_12",
    # "PP_lev_20": "pressure_level_20",
    # "PP_lev_27": "pressure_level_27",
    # "PP_lev_31": "pressure_level_31",
    # "PP_lev_39": "pressure_level_39",
    # "PP_lev_45": "pressure_level_45",
    # "PP_lev_60": "pressure_level_60",
    # # Upper air variables - Temperature
    # "T_lev_6": "temperature_level_6",
    # "T_lev_12": "temperature_level_12",
    # "T_lev_20": "temperature_level_20",
    # "T_lev_27": "temperature_level_27",
    # "T_lev_31": "temperature_level_31",
    # "T_lev_39": "temperature_level_39",
    # "T_lev_45": "temperature_level_45",
    # "T_lev_60": "temperature_level_60",
    # # Upper air variables - Relative Humidity
    # "RELHUM_lev_6": "relative_humidity_level_6",
    # "RELHUM_lev_12": "relative_humidity_level_12",
    # "RELHUM_lev_20": "relative_humidity_level_20",
    # "RELHUM_lev_27": "relative_humidity_level_27",
    # "RELHUM_lev_31": "relative_humidity_level_31",
    # "RELHUM_lev_39": "relative_humidity_level_39",
    # "RELHUM_lev_45": "relative_humidity_level_45",
    # "RELHUM_lev_60": "relative_humidity_level_60",
    # # Upper air variables - Vertical velocity
    # "W_lev_6": "vertical_velocity_level_6",
    # "W_lev_12": "vertical_velocity_level_12",
    # "W_lev_20": "vertical_velocity_level_20",
    # "W_lev_27": "vertical_velocity_level_27",
    # "W_lev_31": "vertical_velocity_level_31",
    # "W_lev_39": "vertical_velocity_level_39",
    # "W_lev_45": "vertical_velocity_level_45",
    # "W_lev_60": "vertical_velocity_level_60",
}

# Since the default ground_truth is the datastore that was used for model training
# the variables are identical to the VARIABLES_GROUND_TRUTH
VARIABLES_ML = VARIABLES_GROUND_TRUTH

# For the NWP-Forecast only a limited set of variables is available
# These variables are mapped to the same conventional names
# The script is flexible and will only calculate the NWP-metrics for the variables that are available
# The script will not break if some of the variables are not available
VARIABLES_NWP = {
    "wind_u_10m": "wind_u_10m",
    "wind_v_10m": "wind_v_10m",
    "precipitation_1hr": "precipitation",
    "pressure_sea_level": "pressure_sea_level",
    "surface_pressure": "surface_pressure",
    "temperature_2m": "temperature_2m",
}

# These variables are only used for chapter 1, the mapplots.
# They will be plotted for the ground truth, NWP and ML
VARIABLES_BOUNDARY = {
    # # Surface and near-surface variables
    "mean_sea_level_pressure": "pressure_sea_level",
    "2m_temperature": "temperature_2m",
    "10m_u_component_of_wind": "wind_u_10m",
    "10m_v_component_of_wind": "wind_v_10m",
    "surface_pressure": "surface_pressure",
    # Upper air variables - U component
    "u_component_of_wind100hPa": "wind_u_level_6",
    "u_component_of_wind200hPa": "wind_u_level_12",
    "u_component_of_wind400hPa": "wind_u_level_20",
    "u_component_of_wind600hPa": "wind_u_level_27",
    "u_component_of_wind700hPa": "wind_u_level_31",
    "u_component_of_wind850hPa": "wind_u_level_39",
    "u_component_of_wind925hPa": "wind_u_level_45",
    "u_component_of_wind1000hPa": "wind_u_level_60",
    # Upper air variables - V component
    "v_component_of_wind100hPa": "wind_v_level_6",
    "v_component_of_wind200hPa": "wind_v_level_12",
    "v_component_of_wind400hPa": "wind_v_level_20",
    "v_component_of_wind600hPa": "wind_v_level_27",
    "v_component_of_wind700hPa": "wind_v_level_31",
    "v_component_of_wind850hPa": "wind_v_level_39",
    "v_component_of_wind925hPa": "wind_v_level_45",
    "v_component_of_wind1000hPa": "wind_v_level_60",
    # Upper air variables - Temperature
    "temperature100hPa": "temperature_level_6",
    "temperature200hPa": "temperature_level_12",
    "temperature400hPa": "temperature_level_20",
    "temperature600hPa": "temperature_level_27",
    "temperature700hPa": "temperature_level_31",
    "temperature850hPa": "temperature_level_39",
    "temperature925hPa": "temperature_level_45",
    "temperature1000hPa": "temperature_level_60",
    # Upper air variables - Vertical velocity
    "vertical_velocity100hPa": "vertical_velocity_level_6",
    "vertical_velocity200hPa": "vertical_velocity_level_12",
    "vertical_velocity400hPa": "vertical_velocity_level_20",
    "vertical_velocity600hPa": "vertical_velocity_level_27",
    "vertical_velocity700hPa": "vertical_velocity_level_31",
    "vertical_velocity850hPa": "vertical_velocity_level_39",
    "vertical_velocity925hPa": "vertical_velocity_level_45",
    "vertical_velocity1000hPa": "vertical_velocity_level_60",
}

# These variables will be used as `basename` for the vertical profiles.
# Since the input of the zarr archives is expected to have data vars that are 2D in space
# we need some base_name prefix to create the 3D variables
VARIABLES_3D = [
    "temperature_level",
    "wind_u_level",
    "wind_v_level",
    "pressure_level",
    "relative_humidity_level",
    "vertical_velocity_level",
]

# Add units dictionary after the imports
# units from zarr archives are not reliable and should rather be defined here
VARIABLE_UNITS = {
    # Surface and near-surface variables
    "temperature_2m": "K",
    "wind_u_10m": "m/s",
    "wind_v_10m": "m/s",
    "pressure_sea_level": "Pa",
    "surface_pressure": "Pa",
    "precipitation": "mm/h",
    "surface_sensible_heat_flux": "W/m²",
    "surface_net_shortwave_radiation": "W/m²",
    "surface_net_longwave_radiation": "W/m²",
    # Upper air variables
    "temperature_level": "K",
    "wind_u_level": "m/s",
    "wind_v_level": "m/s",
    "pressure_level": "Pa",
    "relative_humidity_level": "%",
    "vertical_velocity_level": "Pa/s",
}

# Define Thresholds for the ETS metric (Equitable Threat Score)
# These are calculated for wind and precipitation if available
# The score creates contingency tables for different thresholds
# The ETS is calculated for each threshold and the results are plotted
# The default thresholds are [0.1, 1, 5] for precipitation and [2.5, 5, 10] for wind
THRESHOLDS_PRECIPITATION = [0.1, 1, 5]  # mm/h
THRESHOLDS_WIND = [2.5, 5, 10]  # m/s

# Define the metrics to compute for the verification
# Some additional verifications will always be computed if the repsective vars
# are available in the data
METRICS = [
    # "MAE",
    # "RMSE",
    # "MSE",
    # "ME",
    # "STDEV_ERR",
    # "RelativeMAE",
    # "RelativeRMSE",
    # "PearsonR",
    # "Wasserstein",
    "FSS"
]

# This setting is relevant for the mapplots in chapter 1
# Higher levels of ZOOM will zoom in on the map, cropping the boundary
ZOOM = 2.2

# For some chapters a random seed is required to reproduce the results
RANDOM_SEED = 42

# The DPI used in all plots in the notebook, export to pdf will always be 300 DPI
DPI = 100

# If the data should be loaded into memory. Makes following calculations faster
# but requires enough memory to hold the data.
PRECOMPUTE_DATA = True

# Takes a long time, but if you see NaN in your output, you can set this to True
# This will check if there are any missing values in the data further below
# THIS NOTEBOOK WILL ONLY WORK RELIABLY IF THERE ARE NO MISSING VALUES
# If there are missing values, you have to interpolate them or remove them
CHECK_MISSING = False

# Font sizes for consistent plotting (different fig-sizes wil require different font sizes)
FONT_SIZES = {
    "axes": 20,  # Axis labels and titles
    "ticks": 20,  # Tick labels
    "legend": 18,  # Legend text
    "cbar": 20,  # Colorbar labels
    "suptitle": 20,  # Figure suptitle
    "title": 20,  # Axes titles
    "stats": 18,  # Statistics text in plots
}


In [3]:
# Create directories for plots and tables
Path("plots").mkdir(exist_ok=True)
Path("tables").mkdir(exist_ok=True)
Path(ANIMATION_DIR).mkdir(exist_ok=True)

# Colorblind-friendly color palette
COLORS = {
    "gt": "#000000",  # Black
    "ml": "#E69F00",  # Orange
    "nwp": "#56B4E9",  # Light blue
    "error": "#CC79A7",  # Pink
}

# Line styles and markers for accessibility
LINE_STYLES = {
    "gt": ("solid", "o"),
    "ml": ("dashed", "s"),
    "nwp": ("dotted", "^"),
}

# Set global font sizes
plt.rcParams.update({
    "font.size": FONT_SIZES["axes"],
    "axes.titlesize": FONT_SIZES["axes"],
    "axes.labelsize": FONT_SIZES["axes"],
    "xtick.labelsize": FONT_SIZES["ticks"],
    "ytick.labelsize": FONT_SIZES["ticks"],
    "legend.fontsize": FONT_SIZES["legend"],
    "figure.titlesize": FONT_SIZES["suptitle"],
})

# Colorblind-friendly colormap for 2D plots
COLORMAP = "viridis"

# Add level-specific units by reusing base units
required_levels = {
    int(key.split("_")[-1]) for key in VARIABLES_GROUND_TRUTH if "lev_" in key
}

# First, collect all the base variables and units we need to extend
base_level_vars = {}
for base_var, unit in VARIABLE_UNITS.items():
    if "_level" in base_var:
        base_level_vars[base_var] = unit

# Then create the level-specific entries
for level in required_levels:
    for base_var, unit in base_level_vars.items():
        VARIABLE_UNITS[f"{base_var}_{level}"] = unit


def save_plot(fig, name, time=None, remove_title=True, dpi=300):
    """Helper function to save plots consistently

    Args:
        fig: matplotlib figure object
        name (str): base name for the plot file
        time (datetime, optional): timestamp to append to filename
        remove_title (bool): remove suptitle/title hierarchically if True
        dpi (int): resolution for the saved figure, defaults to 300
    """
    if time is not None:
        name = f"{name}_{time.dt.strftime('%Y%m%d_%H').values}"

    # Sanitize filename by replacing problematic characters
    safe_name = name.replace("/", "_per_")

    # Normalize the path and ensure plots directory exists
    plot_dir = Path("plots")
    plot_dir.mkdir(exist_ok=True)

    # Remove titles if requested
    if remove_title:
        if hasattr(fig, "texts") and fig.texts:  # Check for suptitle
            fig.suptitle("")
        ax = fig.gca()
        if ax.get_title():
            ax.set_title("")

    pdf_path = plot_dir / f"{safe_name}.pdf"
    fig.savefig(pdf_path, bbox_inches="tight", dpi=dpi)


def export_table(df, name, caption=""):
    """Helper function to export tables consistently"""
    # Export to LaTeX with caption
    latex_str = df.to_latex(
        float_format="%.4f", caption=caption, label=f"tab:{name}"
    )
    with open(f"tables/{name}.tex", "w") as f:
        f.write(latex_str)

    # Export to CSV
    df.to_csv(f"tables/{name}.csv")

In [4]:
ds_ml = xr.open_zarr(PATH_ML)
ds_ml = ds_ml.sel(state_feature=list(VARIABLES_ML.keys()))
ds_ml = ds_ml.sel(y=slice(*Y), x=slice(*X))
ds_ml = ds_ml.sel(start_time=slice(*START_TIMES))
for feature in ds_ml.state_feature.values:
    ds_ml[VARIABLES_ML[feature]] = ds_ml["state"].sel(state_feature=feature)
forecast_times = (
    ds_ml.start_time.values[:, None] + ds_ml.elapsed_forecast_duration.values
)
ds_ml = ds_ml.assign_coords(
    forecast_time=(
        ("start_time", "elapsed_forecast_duration"),
        forecast_times,
    )
)
ds_ml = ds_ml.drop_vars(["state", "state_feature", "time"])
ds_ml = ds_ml.transpose("start_time", "elapsed_forecast_duration", "x", "y")
ds_ml = ds_ml[
    [
        "start_time",
        "elapsed_forecast_duration",
        "x",
        "y",
        *VARIABLES_ML.values(),
    ]
]
ds_ml = ds_ml.isel(elapsed_forecast_duration=ELAPSED_FORECAST_DURATION)

ds_ml

<xarray.Dataset> Size: 1GB
Dimensions:                         (start_time: 5,
                                     elapsed_forecast_duration: 119, x: 582,
                                     y: 390)
Coordinates:
  * start_time                      (start_time) datetime64[ns] 40B 2020-02-0...
  * elapsed_forecast_duration       (elapsed_forecast_duration) timedelta64[ns] 952B ...
  * x                               (x) int64 5kB 0 1 2 3 4 ... 578 579 580 581
  * y                               (y) int64 3kB 0 1 2 3 4 ... 386 387 388 389
    forecast_time                   (start_time, elapsed_forecast_duration) datetime64[ns] 5kB ...
Data variables:
    wind_u_10m                      (start_time, elapsed_forecast_duration, x, y) float32 540MB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>
    surface_net_longwave_radiation  (start_time, elapsed_forecast_duration, x, y) float32 540MB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>

In [5]:
ds_gt = xr.open_zarr(PATH_GROUND_TRUTH)
ds_gt = ds_gt.set_index(grid_index=["y", "x"]).unstack("grid_index")
ds_gt = ds_gt.sel(y=slice(*Y), x=slice(*X))
ds_gt = ds_gt.sel(state_feature=list(VARIABLES_ML.keys()))
ds_gt = ds_gt.sel(split_name="test").drop_dims([
    # "forcing_feature",
    # "static_feature",
    "split_part",
])
for feature in ds_gt.state_feature.values:
    ds_gt[VARIABLES_ML[feature]] = ds_gt["state"].sel(state_feature=feature)
ds_gt = ds_gt.drop_vars([
    "state",
    "state_feature",
    "state_feature_units",
    "state_feature_long_name",
    "state_feature_source_dataset",
    "state__train__diff_mean",
    "state__train__diff_std",
    "state__train__mean",
    "state__train__std",
])
ds_gt = ds_gt.transpose("time", "x", "y")
ds_gt = ds_gt[
    [
        "time",
        "x",
        "y",
        *VARIABLES_GROUND_TRUTH.values(),
    ]
]
ds_gt = ds_gt.sel(time=np.unique(ds_ml.forecast_time.values.flatten()))
ds_gt

<xarray.Dataset> Size: 307MB
Dimensions:                         (time: 167, x: 582, y: 390)
Coordinates:
  * time                            (time) datetime64[ns] 1kB 2020-02-08T01:0...
  * x                               (x) int64 5kB 0 1 2 3 4 ... 578 579 580 581
  * y                               (y) int64 3kB 0 1 2 3 4 ... 386 387 388 389
    split_name                      <U5 20B 'test'
    lat                             (x, y) float64 2MB dask.array<chunksize=(582, 390), meta=np.ndarray>
    lon                             (x, y) float64 2MB dask.array<chunksize=(582, 390), meta=np.ndarray>
Data variables:
    wind_u_10m                      (time, x, y) float32 152MB dask.array<chunksize=(1, 582, 390), meta=np.ndarray>
    surface_net_longwave_radiation  (time, x, y) float32 152MB dask.array<chunksize=(1, 582, 390), meta=np.ndarray>
Attributes:
    created_on:       2025-08-18T11:21:21
    created_with:     mllam-data-prep (https://github.com/mllam/mllam-data-prep)
    dataset_version:  v0.1.0
    mdp_version:      v0.5.0
    schema_version:   v0.6.0

In [6]:
ds_nwp = xr.open_zarr(PATH_NWP)
ds_nwp = ds_nwp.sel(y=slice(*Y), x=slice(*X), time=slice(*START_TIMES))
ds_nwp = ds_nwp[VARIABLES_NWP.keys()].rename(VARIABLES_NWP)
ds_nwp = ds_nwp.rename_dims({
    "lead_time": "elapsed_forecast_duration",
    "time": "start_time",
})
ds_nwp = ds_nwp.rename_vars({
    "lead_time": "elapsed_forecast_duration",
    "time": "start_time",
})
forecast_times = (
    ds_nwp.start_time.values[:, None] + ds_nwp.elapsed_forecast_duration.values
)
ds_nwp = ds_nwp.assign_coords(
    forecast_time=(
        ("start_time", "elapsed_forecast_duration"),
        forecast_times,
    )
)

# # Calculate hourly values by taking differences along elapsed_forecast_duration
ds_nwp["precipitation"] = ds_nwp.precipitation.diff(
    dim="elapsed_forecast_duration"
)
# The NWP data starts at elapsed forecast duration 0 = start_time
ds_nwp = ds_nwp.drop_isel(elapsed_forecast_duration=0).isel(
    elapsed_forecast_duration=ELAPSED_FORECAST_DURATION
)

ds_nwp = ds_nwp.transpose("start_time", "elapsed_forecast_duration", "x", "y")
ds_nwp = ds_nwp[
    [
        "start_time",
        "elapsed_forecast_duration",
        "x",
        "y",
        *VARIABLES_NWP.values(),
    ]
]
chunks = {
    "start_time": 1,
    "elapsed_forecast_duration": 1,
    "x": -1,  # or ds_nwp.sizes["x"]
    "y": -1,  # or ds_nwp.sizes["y"]
}

# Create encoding dict for each variable
encoding = {}
for var in ds_nwp.data_vars:
    # Get the actual dimensions for this variable
    dims = ds_nwp[var].dims
    # Create chunks tuple matching the variable's dimensions
    var_chunks = tuple(chunks.get(dim, -1) for dim in dims)
    encoding[var] = {"chunks": var_chunks}

# First ensure Dask chunks match
ds_nwp = ds_nwp.chunk(chunks)

ds_nwp

<xarray.Dataset> Size: 6GB
Dimensions:                    (start_time: 5, elapsed_forecast_duration: 119,
                                x: 582, y: 390)
Coordinates:
  * start_time                 (start_time) datetime64[ns] 40B 2020-02-08 ......
  * elapsed_forecast_duration  (elapsed_forecast_duration) timedelta64[ns] 952B ...
  * x                          (x) int64 5kB 0 1 2 3 4 5 ... 577 578 579 580 581
  * y                          (y) int64 3kB 0 1 2 3 4 5 ... 385 386 387 388 389
    lat                        (x, y) float64 2MB dask.array<chunksize=(582, 390), meta=np.ndarray>
    lon                        (x, y) float64 2MB dask.array<chunksize=(582, 390), meta=np.ndarray>
    forecast_time              (start_time, elapsed_forecast_duration) datetime64[ns] 5kB dask.array<chunksize=(1, 1), meta=np.ndarray>
Data variables:
    wind_u_10m                 (start_time, elapsed_forecast_duration, x, y) float64 1GB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>
    wind_v_10m                 (start_time, elapsed_forecast_duration, x, y) float64 1GB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>
    precipitation              (start_time, elapsed_forecast_duration, x, y) float64 1GB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>
    pressure_sea_level         (start_time, elapsed_forecast_duration, x, y) float64 1GB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>
    surface_pressure           (start_time, elapsed_forecast_duration, x, y) float64 1GB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>
    temperature_2m             (start_time, elapsed_forecast_duration, x, y) float64 1GB dask.array<chunksize=(1, 1, 582, 390), meta=np.ndarray>

Check for missing data in any of the variables. If you have missing data, you need to handle it before running the verification.

In [7]:
if CHECK_MISSING:
    missing_counts = dask.compute(
        {var: ds_gt[var].isnull().sum().values for var in ds_gt.data_vars},
        {var: ds_nwp[var].isnull().sum().values for var in ds_nwp.data_vars},
        {var: ds_ml[var].isnull().sum().values for var in ds_ml.data_vars},
    )
    # Unpack results
    gt_missing, nwp_missing, ml_missing = missing_counts

    # Print results
    print("Ground Truth")
    for var, count in gt_missing.items():
        print(f"{var}: {count} missing values")

    print("\nNWP Model")
    for var, count in nwp_missing.items():
        print(f"{var}: {count} missing values")

    print("\nML Model")
    for var, count in ml_missing.items():
        print(f"{var}: {count} missing values")

In [8]:
assert ds_gt.sizes["x"] == ds_ml.sizes["x"]
assert ds_gt.sizes["x"] == ds_nwp.sizes["x"]
assert ds_gt.sizes["y"] == ds_ml.sizes["y"]
assert ds_gt.sizes["y"] == ds_nwp.sizes["y"]
assert int(ds_gt.sizes["time"]) == len(
    np.unique(ds_ml.forecast_time.values.flatten())
)
assert int(ds_gt.sizes["time"]) == len(
    np.unique(ds_nwp.forecast_time.values.flatten())
)


### 1. Maps

**Random Time Selection:** A random time step is selected to avoid bias in the comparison, ensuring that the assessment is representative of typical model performance.

**Consistent Color Scales:** By setting the same minimum and maximum values across all datasets for each variable, we ensure that color differences in the plots reflect true discrepancies, not artifacts of scaling.

**Spatial Patterns:** The plots reveal how the ML model and NWP model represent geographical features like weather fronts, high and low-pressure systems, and temperature gradients. Visual comparisons can immediately highlight areas where the models perform well or poorly, guiding further investigation.

**Edge Effects:** Near the boundaries, artifacts may occur as the model does not calculate a loss in the boundary region.

In [9]:
# Get coordinates
if hasattr(ds_gt, "longitude") and hasattr(ds_gt, "latitude"):
    lons = ds_gt.longitude.values
    lats = ds_gt.latitude.values
elif hasattr(ds_gt, "lon") and hasattr(ds_gt, "lat"):
    lons = ds_gt.lon.values
    lats = ds_gt.lat.values
lon_min = lons.min()
lon_max = lons.max()
lat_min = lats.min()
lat_max = lats.max()

# Transform domain bounds to rotated coordinates
transformer = PROJECTION.transform_points(
    ccrs.PlateCarree(),
    np.array([lon_min, lon_max]),
    np.array([lat_min, lat_max]),
)

# Get rotated coordinate bounds
rot_lon_min, rot_lon_max = transformer[:, 0].min(), transformer[:, 0].max()
rot_lat_min, rot_lat_max = transformer[:, 1].min(), transformer[:, 1].max()

In [10]:
ds_boundary = xr.open_zarr(PATH_BOUNDARY)

temporal_dim = "time" if "time" in ds_boundary.dims else "analysis_time"
forecast_duration_dim = (
    "elapsed_forecast_duration"
    if "elapsed_forecast_duration" in ds_boundary.dims
    else None
)
dims_to_transpose = [
    dim
    for dim in [temporal_dim, forecast_duration_dim, "latitude", "longitude"]
    if dim is not None
]

ds_boundary = ds_boundary.sel(forcing_feature=list(VARIABLES_BOUNDARY.keys()))
ds_boundary = ds_boundary.sel(split_name="test").drop_dims([
    "split_part",
    # "static_feature",
])
for feature in ds_boundary.forcing_feature.values:
    ds_boundary[VARIABLES_BOUNDARY[feature]] = ds_boundary["forcing"].sel(
        forcing_feature=feature
    )
ds_boundary = ds_boundary.drop_vars([
    "forcing",
    "forcing_feature",
    "forcing_feature_units",
    "forcing_feature_long_name",
    "forcing_feature_source_dataset",
    "forcing__train__diff_mean",
    "forcing__train__diff_std",
    "forcing__train__mean",
    "forcing__train__std",
])
ds_boundary = ds_boundary.set_index(grid_index=["latitude", "longitude"])
ds_boundary = ds_boundary.unstack("grid_index")
ds_boundary = ds_boundary.transpose(*dims_to_transpose)
longitude_new = np.where(
    ds_boundary["longitude"] > 180,
    ds_boundary["longitude"] - 360,
    ds_boundary["longitude"],
)
ds_boundary = ds_boundary.assign_coords(longitude=longitude_new).sortby([
    "longitude",
    "latitude",
])


lon_mesh, lat_mesh = np.meshgrid(ds_boundary.longitude, ds_boundary.latitude)
ds_boundary

/users/sadamov/miniforge3/envs/neural-lam/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 17
  result = blockwise(
/users/sadamov/miniforge3/envs/neural-lam/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 17
  result = blockwise(
/users/sadamov/miniforge3/envs/neural-lam/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 17
  result = blockwise(
/users/sadamov/miniforge3/envs/neural-lam/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 17
  result = blockwise(
/users/sadamov/miniforge3/envs/neural-lam/lib/python3.12/site-packages/dask/array/core.py:5092: PerformanceWarning: Increasing number of chunks by factor of 17
  result = blockwise(
/users/sadamov/miniforge3/envs/neural-lam/lib/python3.12/site-packages/dask/array/core.py:

<xarray.Dataset> Size: 2GB
Dimensions:                     (latitude: 91, analysis_time: 17,
                                 elapsed_forecast_duration: 41, longitude: 162)
Coordinates:
  * latitude                    (latitude) float32 364B 35.0 35.25 ... 57.5
  * analysis_time               (analysis_time) datetime64[ns] 136B 2020-02-0...
  * elapsed_forecast_duration   (elapsed_forecast_duration) timedelta64[ns] 328B ...
    split_name                  <U5 20B 'test'
  * longitude                   (longitude) float32 648B -11.75 -11.5 ... 28.5
Data variables: (12/37)
    pressure_sea_level          (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    temperature_2m              (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    wind_u_10m                  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    wind_v_10m                  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    surface_pressure            (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    wind_u_level_6              (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    ...                          ...
    vertical_velocity_level_20  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    vertical_velocity_level_27  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    vertical_velocity_level_31  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    vertical_velocity_level_39  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    vertical_velocity_level_45  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
    vertical_velocity_level_60  (analysis_time, elapsed_forecast_duration, latitude, longitude) float32 41MB dask.array<chunksize=(1, 1, 91, 162), meta=np.ndarray>
Attributes:
    created_on:       2025-08-18T11:34:53
    created_with:     mllam-data-prep (https://github.com/mllam/mllam-data-prep)
    dataset_version:  v0.1.0
    mdp_version:      v0.5.0
    schema_version:   v0.5.0

In [11]:
def create_comparison_gifs(
    ds_gt,
    ds_ml,
    ds_nwp,
    ds_boundary=None,
    var=None,
    plot_time=None,
    step_size_gt=pd.Timedelta("1h"),
    random_seed=42,
    zoom_factor=None,
):
    # Handle variable selection
    variables = [var] if var else VARIABLES_GROUND_TRUTH.values()

    # Select time
    if plot_time is None:
        random.seed(random_seed)
        time_index = random.randint(0, len(ds_gt.time) - 1)
        time_selected = ds_ml.time[time_index].values
    else:
        time_selected = plot_time

    for var in variables:
        # Determine number of columns based on NWP data availability
        n_cols = 3 if (ds_nwp is not None and var in ds_nwp) else 2

        # Initialize arrays for global min/max
        arrays_for_minmax = []

        # First pass: collect all data for global min/max calculation
        for elapsed_forecast_dimension in ds_ml.elapsed_forecast_duration:
            # Select data for current forecast time
            ds_ml_time = ds_ml.sel(
                start_time=time_selected,
                elapsed_forecast_duration=elapsed_forecast_dimension,
            )
            ds_gt_time = ds_gt.sel(time=ds_ml_time.forecast_time)

            # Add ground truth and ML prediction to min/max arrays
            arrays_for_minmax.extend([
                ds_gt_time[var].values,
                ds_ml_time[var].values,
            ])

            # Add NWP data if available
            if (
                ds_nwp is not None
                and var in ds_nwp
                and elapsed_forecast_dimension.values
                in ds_nwp.elapsed_forecast_duration
            ):
                ds_nwp_time = ds_nwp.sel(
                    start_time=time_selected,
                    elapsed_forecast_duration=elapsed_forecast_dimension,
                )
                arrays_for_minmax.append(ds_nwp_time[var].values)

            # Add boundary data if available
            if ds_boundary is not None and var in ds_boundary:
                if "elapsed_forecast_duration" in ds_boundary:
                    # Determine number of steps based on time alignment
                    steps = (
                        2
                        if (
                            ds_boundary.sel(
                                analysis_time=time_selected - step_size_gt,
                                method="pad",
                            ).analysis_time.values
                            == time_selected - step_size_gt
                        )
                        else 1
                    )

                    # Get boundary data for current time
                    ds_boundary_var = ds_boundary[var].sel(
                        analysis_time=time_selected - steps * step_size_gt,
                        method="pad",
                    )
                    # Update forecast times
                    forecast_times = (
                        ds_boundary_var.analysis_time.values
                        + ds_boundary_var.elapsed_forecast_duration.values
                    )
                    ds_boundary_var["elapsed_forecast_duration"] = (
                        forecast_times
                    )
                    ds_boundary_var = ds_boundary_var.sel(
                        elapsed_forecast_duration=ds_ml_time.forecast_time,
                        method="pad",
                    )
                else:
                    ds_boundary_var = ds_boundary[var].sel(
                        time=ds_ml_time.forecast_time, method="pad"
                    )
                arrays_for_minmax.append(ds_boundary_var.values)

        # Calculate global min/max
        combined_array = np.concatenate([
            arr.flatten() for arr in arrays_for_minmax
        ])
        vmin, vmax = np.nanmin(combined_array), np.nanmax(combined_array)

        # Create list to store figure images in
        frame_list = []

        # Second pass: create plots
        for dim_idx, elapsed_forecast_dimension in enumerate(
            ds_ml.elapsed_forecast_duration
        ):
            # Select data for current forecast time
            ds_ml_time = ds_ml.sel(
                start_time=time_selected,
                elapsed_forecast_duration=elapsed_forecast_dimension,
            )
            ds_gt_time = ds_gt.sel(time=ds_ml_time.forecast_time)

            # Calculate forecast hours for titles
            forecast_hours = int(elapsed_forecast_dimension.values / 1e9 / 3600)

            # Set up figure
            # Create figure with n_elapsed_forecast_durations rows and n_cols columns
            fig = plt.figure(figsize=(7 * n_cols, 7), dpi=DPI)
            axes = np.array([
                plt.subplot(
                    1,
                    n_cols,
                    j + 1,
                    projection=PROJECTION,
                )
                for j in range(n_cols)
            ])

            # Plot boundary conditions if available
            if ds_boundary is not None and var in ds_boundary:
                if "elapsed_forecast_duration" in ds_boundary:
                    steps = (
                        2
                        if (
                            ds_boundary.sel(
                                analysis_time=time_selected - step_size_gt,
                                method="pad",
                            ).analysis_time.values
                            == time_selected - step_size_gt
                        )
                        else 1
                    )

                    ds_boundary_var = ds_boundary[var].sel(
                        analysis_time=time_selected - steps * step_size_gt,
                        method="pad",
                    )
                    forecast_times = (
                        ds_boundary_var.analysis_time.values
                        + ds_boundary_var.elapsed_forecast_duration.values
                    )
                    ds_boundary_var["elapsed_forecast_duration"] = (
                        forecast_times
                    )
                    ds_boundary_var = ds_boundary_var.sel(
                        elapsed_forecast_duration=ds_ml_time.forecast_time,
                        method="pad",
                    )
                else:
                    ds_boundary_var = ds_boundary[var].sel(
                        time=ds_ml_time.forecast_time, method="pad"
                    )

                # Define fixed contour levels based on global min/max
                n_levels = 50
                fixed_levels = np.linspace(vmin, vmax, n_levels + 1)

                for ax in axes:
                    ax.contourf(
                        lon_mesh,
                        lat_mesh,
                        ds_boundary_var.values,
                        transform=ccrs.PlateCarree(),
                        cmap="viridis",
                        vmin=vmin,
                        vmax=vmax,
                        alpha=0.5,
                        levels=fixed_levels,  # Use fixed levels
                    )
                    if zoom_factor is not None:
                        set_map_extent(ax, zoom_factor, ds_boundary_var)

            # Plot ground truth
            im0 = plot_field(axes[0], ds_gt_time[var], vmin, vmax)

            # Set titles
            axes[0].set_title(f"Ground Truth\n+{forecast_hours}h")
            if ds_nwp is not None and var in ds_nwp:
                axes[1].set_title(f"NWP\n+{forecast_hours}h")
                axes[2].set_title(f"ML\n+{forecast_hours}h")
            else:
                axes[1].set_title(f"ML\n+{forecast_hours}h")

            # Plot NWP and ML predictions
            col = 1
            if ds_nwp is not None and var in ds_nwp:
                # Do not plot anything for NWP if beyond prediction time
                if (
                    elapsed_forecast_dimension.values
                    in ds_nwp.elapsed_forecast_duration
                ):
                    ds_nwp_time = ds_nwp.sel(
                        start_time=time_selected,
                        elapsed_forecast_duration=elapsed_forecast_dimension,
                    )
                    plot_field(axes[col], ds_nwp_time[var], vmin, vmax)
                col += 1

            plot_field(axes[col], ds_ml_time[var], vmin, vmax)

            # Add common features and colorbar
            add_map_features(axes)
            add_colorbar(fig, im0, var)

            # Adjust layout and add title
            # plt.subplots_adjust(
            #    top=0.92,
            #    bottom=0.05,
            #    hspace=0.2,
            #    wspace=0.05,
            # )
            title = f"{var} starting at {str(time_selected.dt.date.values)} - {time_selected.dt.hour.values:02d} UTC"
            plt.suptitle(title, y=0.98)

            # Store figure
            fig.canvas.draw()  # Required
            fig_pil = PIL.Image.frombytes(
                "RGBa", fig.canvas.get_width_height(), fig.canvas.buffer_rgba()
            ).convert("RGB")
            frame_list.append(fig_pil)
            plt.close()

        # Make gif
        anim_fname = f"map_{var}.gif"
        frame_list[0].save(
            os.path.join(ANIMATION_DIR, anim_fname),
            format="GIF",
            append_images=frame_list[1:],
            save_all=True,
            duration=ANIMATION_DURATION,
            loop=0,
        )
        print(f"Saved {anim_fname}")

        # plt.show()
        # save_plot(fig, f"map_{var}_multi_efd", time_selected, dpi=DPI)


def set_map_extent(ax, zoom_factor=None, boundary_data=None):
    """Set the map extent based on zoom factor and boundary data."""
    if zoom_factor is not None and boundary_data is not None:
        # Get the boundary extent
        lon = (
            boundary_data.longitude
            if hasattr(boundary_data, "longitude")
            else boundary_data.lon
        )
        lat = (
            boundary_data.latitude
            if hasattr(boundary_data, "latitude")
            else boundary_data.lat
        )

        # Calculate center
        lon_center = (lon.max() + lon.min()) / 2
        lat_center = (lat.max() + lat.min()) / 2

        # Calculate ranges
        lon_range = (lon.max() - lon.min()) / zoom_factor
        lat_range = (lat.max() - lat.min()) / zoom_factor

        # Set new extent
        ax.set_extent(
            [
                lon_center - lon_range / 2,
                lon_center + lon_range / 2,
                lat_center - lat_range / 2,
                lat_center + lat_range / 2,
            ],
            crs=ccrs.PlateCarree(),
        )


def plot_field(ax, data, vmin, vmax):
    return ax.pcolormesh(
        data.longitude if hasattr(data, "longitude") else data.lon,
        data.latitude if hasattr(data, "latitude") else data.lat,
        data.values,
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax,
        cmap="viridis",
        shading="auto",
        rasterized=True,
    )


def add_map_features(axes):
    for j, ax in enumerate(axes):
        ax.coastlines(resolution="50m")
        ax.add_feature(cfeature.BORDERS, linestyle="-", alpha=0.7)
        gl = ax.gridlines(
            draw_labels=True,
            dms=True,
            x_inline=False,
            y_inline=False,
            rotate_labels=False,
        )

        # Turn off all labels by default
        gl.top_labels = False
        gl.bottom_labels = False
        gl.left_labels = False
        gl.right_labels = False

        # Enable left labels only for leftmost column
        if j == 0:
            gl.left_labels = True

        gl.bottom_labels = True


def add_colorbar(fig, im, var):
    cbar_ax = fig.add_axes([0.2, 0.12, 0.6, 0.02])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(VARIABLE_UNITS[var])


In [12]:
step_size_gt = pd.Timedelta(ds_gt.time.diff("time").min().values, "h")
ds_ml = ds_ml.assign_coords({
    "lon": (("x", "y"), ds_nwp.lon.values),
    "lat": (("x", "y"), ds_nwp.lat.values),
})
if PLOT_TIME is None:
    time_selected = None
else:
    time_selected = ds_ml.sel(start_time=PLOT_TIME).start_time
create_comparison_gifs(
    ds_gt=ds_gt,
    ds_ml=ds_ml,
    ds_nwp=ds_nwp,
    ds_boundary=ds_boundary,
    plot_time=time_selected,
    step_size_gt=step_size_gt,
    zoom_factor=ZOOM,
)


Saved map_wind_u_10m.gif
Saved map_surface_net_longwave_radiation.gif
